# Course 06 lab — Writing executable requirements

Turn the AI-2219 broker-response story into mutually consistent requirements, decision rules, scenarios, contracts, states, and evidence. The notebook is credential-free: a model proposal is fixture data, never authority.

## 1. Load the deterministic teaching module

Run this notebook from the repository root or from this course directory.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

candidates = [
    Path.cwd() / 'lab.py',
    Path.cwd() / 'curriculum/beginner/06-writing-executable-requirements/lab.py',
]
lab_path = next(path for path in candidates if path.exists())
spec = importlib.util.spec_from_file_location('course06_lab', lab_path)
lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
print(f'Loaded {lab_path}')

## 2. Inspect artifact authority before behavior

Machine-readable does not mean authoritative. The contract explicitly declares which representations are normative, explanatory, or evidence.

In [ ]:
contract = lab.load_contract()
print(json.dumps(contract['artifact_authority'], indent=2))
print('Capabilities:')
for item in contract['capabilities']:
    print(f"- {item['id']}: {item['status']}")

## 3. Diagnose requirement language

Lint is diagnostic. It can identify risky wording, but it cannot decide whether a requirement is correct, feasible, authorized, or complete. `POSSIBLE_AMBIGUOUS_REFERENT` is intentionally a review signal rather than a semantic verdict.

In [ ]:
weak = {
    'id': 'EX-AMBIGUOUS',
    'statement': 'The system SHALL quickly handle every field and/or update it appropriately, etc.',
    'check_pronouns': True,
}
for finding in lab.writing_findings(weak):
    print(f'{finding.severity.value.upper():6} {finding.code}: {finding.message}')

clean_findings = lab.requirement_findings(contract)
print(f'Reference requirement findings: {len(clean_findings)}')

## 4. Inspect EARS forms and full behavioral fields

The SHALL sentence is the readable center. Preconditions, postconditions, frame conditions, failure behavior, ownership, and evidence make its boundary inspectable.

In [ ]:
for requirement in contract['requirements']:
    if requirement['id'] in {'REQ-BR-001', 'REQ-BR-003', 'REQ-BR-005', 'REQ-BR-006', 'REQ-BR-007', 'REQ-BR-020', 'REQ-BR-021', 'REQ-BR-037'}:
        print(f"{requirement['id']} · {requirement['ears_pattern']}")
        print(requirement['statement'])
        print('frame:', '; '.join(requirement['frame_conditions']))
        print()

## 5. Exercise the normative decision table

A supported fact combination must match exactly one row. Row order is not a conflict-resolution mechanism.

In [ ]:
verified_conflict = {
    'existing_value': 'present',
    'existing_verified': True,
    'proposed_valid': True,
    'same_value': False,
    'auto_accept_allowed': True,
}
outcome, row_id = lab.decision_table_outcome(verified_conflict)
print({'row_id': row_id, 'outcome': outcome})
assert outcome == 'conflict'

## 6. Inject a semantic contradiction

The mutation weakens the verified-conflict outcome. The correct result is STOP, not silent selection of the prose, table, scenario, or easiest implementation.

In [ ]:
mutated = lab.mutated_table()
findings = lab.artifact_consistency_findings(table=mutated)
for finding in findings:
    print(f'{finding.severity.value.upper()} {finding.code}: {finding.message}')
assert any(item.code == 'NORMATIVE_ARTIFACT_CONFLICT' for item in findings)

### 6a. Distinguish incomplete and ambiguous tables

Removing the invalid-value row creates zero matches. Adding an overlapping row creates two matches. Both are STOP findings, but they diagnose different defects.

In [ ]:
incomplete = lab.decision_table_shape_findings(lab.incomplete_table())
ambiguous = lab.decision_table_shape_findings(lab.ambiguous_table())
print('Incomplete codes:', sorted({item.code for item in incomplete}))
print('Ambiguous codes:', sorted({item.code for item in ambiguous}))
assert any(item.code == 'DECISION_TABLE_INCOMPLETE' for item in incomplete)
assert any(item.code == 'DECISION_TABLE_AMBIGUOUS' for item in ambiguous)

## 7. Prove that model status is not authority

This proposal claims it is already applied. Trusted code ignores that label and recomputes a conflict from authoritative facts.

In [ ]:
digest = 'sha256:' + 'a' * 64
proposal = lab.ModelProposal(
    proposal_id='PROP-DEMO', submission_id='SUB-42', submission_revision='S17',
    requirement_context_digest=digest, field_candidates=('construction_year',),
    proposed_value=2001, source_response_id='MSG-42',
    evidence=(lab.SourceEvidence('EV-42', 'MSG-42', 'built in 2001'),),
    requirement_id='REQ-BR-005', model_version='fixture-v1', model_status='applied',
)
context = lab.DecisionContext(
    current_submission_revision='S17', current_requirement_context_digest=digest,
    broker_authorized=True, response_authenticated=True,
    outstanding_fields=frozenset({'construction_year'}),
    supported_fields=frozenset({'construction_year'}),
    automatic_acceptance_fields=frozenset({'construction_year'}),
    existing_fields={'construction_year': lab.ExistingField(1998, True)},
)
decision = lab.classify_proposal(proposal, context)
print(decision)
assert decision.status == lab.ProposalStatus.CONFLICTING

unknown_context = lab.replace(context, existing_fields={'construction_year': lab.ExistingField(1998, None)})
unknown_decision = lab.classify_proposal(proposal, unknown_context)
print('Unknown verification:', unknown_decision)
assert unknown_decision.reason_codes == ('EXISTING_VERIFICATION_UNKNOWN',)

## 8. Inspect explicit guards, review transitions, and graph properties

In [ ]:
machine = lab.load_state_machine()
print('Valid transitions:', len(machine['transitions']))
for item in machine['critical_invalid_transitions']:
    print('FORBIDDEN', item)
print('Can a conflict apply directly?', lab.transition_status(lab.ProposalStatus.CONFLICTING, 'apply', 'always'))
print('Conflict review entry:', lab.transition_status(lab.ProposalStatus.CONFLICTING, 'request_review', 'authenticated_reviewer_assignment'))
print('Replacement guard:', json.dumps(machine['guards']['valid_conflict_replacement_receipt'], indent=2))
print('Graph findings:', lab.state_machine_findings(machine))
assert lab.transition_status(lab.ProposalStatus.CONFLICTING, 'apply', 'always') is None
assert lab.transition_status(lab.ProposalStatus.CONFLICTING, 'request_review', 'authenticated_reviewer_assignment') == lab.ProposalStatus.AWAITING_REVIEW
assert lab.state_machine_findings(machine) == ()

conflict_update = lab.materialize_update(proposal, decision)
approved_conflict = lab.replace(conflict_update, status=lab.ProposalStatus.APPROVED)
generic_receipt = lab.ApprovalReceipt(
    'UW-22', 'approved', 'Reviewed.', lab.proposal_digest(approved_conflict),
    'SUB-42', 'S17', digest, '2026-09-20T20:00:00Z', '2026-09-20T22:00:00Z',
)
now = lab.datetime(2026, 9, 20, 21, 0, tzinfo=lab.timezone.utc)
generic_result = lab.authorize_application(approved_conflict, context, generic_receipt, now=now)
replacement_receipt = lab.replace(generic_receipt, resolution='replace_verified_value')
replacement_result = lab.authorize_application(approved_conflict, context, replacement_receipt, now=now)
print('Generic approval:', generic_result.reason_codes)
print('Explicit replacement:', replacement_result.outcome.value)
assert 'APPROVAL_RESOLUTION_MISMATCH' in generic_result.reason_codes
assert replacement_result.outcome == lab.Outcome.PROCEED

## 9. Check frame conditions at the mutation boundary

Only a trusted authorization decision can reach the in-memory update. The changed-field list is executable evidence for the frame condition.

In [ ]:
safe_proposal = lab.ModelProposal(
    proposal_id='PROP-SAFE', submission_id='SUB-42', submission_revision='S17',
    requirement_context_digest=digest, field_candidates=('construction_year',),
    proposed_value=2001, source_response_id='MSG-SAFE',
    evidence=(lab.SourceEvidence('EV-SAFE', 'MSG-SAFE', 'built in 2001'),),
    requirement_id='REQ-BR-006', model_version='fixture-v1',
)
safe_context = lab.DecisionContext(
    'S17', digest, True, True, frozenset({'construction_year'}),
    frozenset({'construction_year'}), frozenset({'construction_year'}), {},
)
safe_decision = lab.classify_proposal(safe_proposal, safe_context)
update = lab.materialize_update(safe_proposal, safe_decision)
authorization = lab.authorize_application(update, safe_context)
before = {'construction_year': None, 'occupancy': 'office', 'building_value': 900000}
after, changed = lab.apply_in_memory(before, update, authorization)
print({'decision': safe_decision.disposition.value, 'changed': changed, 'after': after})
assert changed == ('construction_year',)

## 10. Report capability-scoped readiness

Readiness is not an arbitrary percentage. Each capability names its exact blockers.

In [ ]:
for item in lab.capability_readiness():
    print({
        'capability': item.capability,
        'ready': item.implementation_ready,
        'declared_status': item.declared_status,
        'blockers': item.blocking_ids,
        'reasons': item.reason_codes,
    })

## 11. Compare an unsafe baseline with governed decisions

The synthetic dataset evaluates application-owned status and disposition after extraction. It does not measure a live model.

In [ ]:
evaluation = lab.evaluate_cases()
print(json.dumps({
    'dataset_id': evaluation['dataset_id'],
    'population': evaluation['population'],
    'baseline': evaluation['baseline'],
    'governed': evaluation['governed'],
    'limitations': evaluation['limitations'],
}, indent=2))

## 12. Explore a property, coverage, and semantic change

In [ ]:
print('Conflict property:', lab.conflict_property())
print('Specification coverage:')
for name, ratio in lab.specification_coverage().items():
    print(f'- {name}: {ratio.numerator}/{ratio.denominator}')

before_requirement = {'id': 'REQ-DEMO', 'required_fields': ['occupancy'], 'failure_behavior': 'review'}
after_requirement = {'id': 'REQ-DEMO', 'required_fields': ['occupancy', 'construction_year'], 'failure_behavior': 'block'}
print('Semantic change:', lab.requirement_change(before_requirement, after_requirement))
print('Impact of REQ-BR-005:', lab.impact_analysis('REQ-BR-005'))

## 13. Production translation worksheet

Before adapting the pattern, fill in this table with accountable owners and authoritative evidence.

| Question | Your answer | Evidence / owner |
| --- | --- | --- |
| What is the governed population? |  |  |
| Which representation is normative? |  |  |
| What facts drive the decision table? |  |  |
| Which states and transitions are prohibited? |  |  |
| What remains unchanged after success or failure? |  |  |
| How are freshness and authorization established? |  |  |
| What exact object and resolution does approval bind? |  |  |
| What is the idempotency and reconciliation strategy? |  |  |
| Which evaluation population and oracle are owned? |  |  |
| What conditions force STOP? |  |  |

## 14. Your exercise

Complete the AI-2219 starter workspace without consulting the reference. Add one high-risk boundary case, one invalid transition, one frame condition, and one semantic change. Then explain which artifact is authoritative for each claim and why your agent cannot expand its own mutation authority.